Configuração inicial

In [14]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None) # Limite de colunas que serão mostradas
pd.set_option("display.max_rows", 100) # Limite de linhas que serão mostradas

plt.rcParams["figure.figsize"] = (10,6) # Tamanho das figuras, largura e altura

Importação da base

In [15]:
df_acre = pd.read_csv("base_dados/MICRODADOS_ENEM_2023_pre-processamento.csv")
print("Base carregada com sucesso.")

Base carregada com sucesso.


Checagem dos valores nulos

In [16]:
df_nulos = df_acre.isnull().sum().sort_values(ascending=False) # Contagem de nulos e ordenação decrescente
df_nulos = df_nulos.to_frame("Quantidade de nulos")
df_nulos["Porcentagem"] = (df_acre.isnull().sum() / len(df_acre) * 100).sort_values(ascending=False) # Coluna de porcentagem

print(df_nulos)

                        Quantidade de nulos  Porcentagem
SG_UF_ESC                             20077    82.709895
NO_MUNICIPIO_ESC                      20077    82.709895
TP_SIT_FUNC_ESC                       20077    82.709895
TP_DEPENDENCIA_ADM_ESC                20077    82.709895
TP_LOCALIZACAO_ESC                    20077    82.709895
CO_MUNICIPIO_ESC                      20077    82.709895
CO_UF_ESC                             20077    82.709895
TP_ENSINO                             17746    73.107028
NU_NOTA_CN                             9120    37.571064
NU_NOTA_MT                             9120    37.571064
CO_PROVA_MT                            9120    37.571064
CO_PROVA_CN                            9120    37.571064
NU_NOTA_COMP3                          8112    33.418472
CO_PROVA_CH                            8112    33.418472
NU_NOTA_CH                             8112    33.418472
NU_NOTA_LC                             8112    33.418472
CO_PROVA_LC                    

# Análise dos dados envolvendo escola

## Mapeamento das colunas de escolas

In [17]:
df_escolas = df_acre.dropna(subset=['TP_DEPENDENCIA_ADM_ESC', 'TP_LOCALIZACAO_ESC']).copy()

mapeamento_escola = {1: 'Pública', 2: 'Pública', 3: 'Pública', 4: 'Privada'}
mapeamento_localizacao = {1: 'Urbana', 2: 'Rural'}
df_escolas['LOCALIZACAO_ESC'] = df_escolas['TP_LOCALIZACAO_ESC'].map(mapeamento_localizacao)
df_escolas['TIPO_ESCOLA_ADM'] = df_escolas['TP_DEPENDENCIA_ADM_ESC'].map(mapeamento_escola)

In [18]:
colunas_notas = ['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO']

for coluna_nota in colunas_notas:
    df_escolas[coluna_nota] = df_escolas[coluna_nota].fillna(df_escolas[coluna_nota].median())

Configuração da coluna de notas

In [19]:
df_escolas['MEDIA_GERAL'] = df_escolas[colunas_notas].mean(axis=1)

print(f"Dataframe apenas com alunos com dados escolares preenchidos:", df_escolas.shape)

Dataframe apenas com alunos com dados escolares preenchidos: (4197, 68)


Comparação entre alunos de escola pública e alunos de escola privada (Média):


In [20]:
comparativo_adm = df_escolas.groupby('TIPO_ESCOLA_ADM')['MEDIA_GERAL'].mean().reset_index()
display(comparativo_adm)

,TIPO_ESCOLA_ADM,MEDIA_GERAL
0,Privada,611.436779
1,Pública,502.102936


Comparação entre alunos de escola em zona rural e zona urbana

In [21]:
comparativo_localizacao = df_escolas.groupby('LOCALIZACAO_ESC')['MEDIA_GERAL'].mean().reset_index()
display(comparativo_localizacao)

,LOCALIZACAO_ESC,MEDIA_GERAL
0,Rural,495.690489
1,Urbana,515.936603


Dados nulos nesse dataframe

In [22]:
df_nulos_escola = df_escolas.isnull().sum().sort_values(ascending=False) # Contagem de nulos e ordenação decrescente
print(df_nulos_escola)

CO_PROVA_MT               1229
CO_PROVA_CN               1229
NU_NOTA_COMP1             1045
CO_PROVA_CH               1045
NU_NOTA_COMP4             1045
NU_NOTA_COMP5             1045
NU_NOTA_COMP2             1045
NU_NOTA_COMP3             1045
TP_STATUS_REDACAO         1045
CO_PROVA_LC               1045
TP_ENSINO                  254
TP_FAIXA_ETARIA              0
TP_COR_RACA                  0
TP_ESTADO_CIVIL              0
TP_SEXO                      0
CO_UF_ESC                    0
TP_ESCOLA                    0
TP_NACIONALIDADE             0
TP_ANO_CONCLUIU              0
TP_ST_CONCLUSAO              0
SG_UF_PROVA                  0
CO_UF_PROVA                  0
CO_MUNICIPIO_PROVA           0
TP_SIT_FUNC_ESC              0
TP_LOCALIZACAO_ESC           0
TP_DEPENDENCIA_ADM_ESC       0
SG_UF_ESC                    0
TP_PRESENCA_MT               0
TP_PRESENCA_CN               0
TP_PRESENCA_CH               0
TP_PRESENCA_LC               0
NU_NOTA_CN                   0
NU_NOTA_

Verificação dos outliers

In [23]:
Q1 = df_escolas["MEDIA_GERAL"].quantile(0.25)
Q3 = df_escolas["MEDIA_GERAL"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - (1.5 * IQR)
upper = Q3 + (1.5 * IQR)

outliers = df_escolas[(df_escolas["MEDIA_GERAL"] < lower) | (df_escolas['MEDIA_GERAL'] > upper)]
print("Outliers em MEDIA_GERAL utilizando IQR:")
print(outliers["MEDIA_GERAL"])


Outliers em MEDIA_GERAL utilizando IQR:
22       724.98
417      188.84
862      682.24
925      713.80
993      684.14
          ...  
22986    331.16
23370    305.66
23554    300.86
23597    329.34
24151    340.42
Name: MEDIA_GERAL, Length: 243, dtype: float64


Mapeamento de colunas do questionário socioeconômico

In [ ]:
# Colunas respectivamente: Escolaridade do pai, escolaridade da mãe, renda, celular, computador e internet.
# ["Q001", "Q002", "Q006","Q022","Q024","Q025"]

mapeamento_escolaridade = {
    'A': 1,
    'B': 2,
    'C': 2,
    'D': 3,
    'E' : 4,
    'F' : 5,
    'G' : 5,
    'H' : 6
}

df_questionario = df_acre.copy()


df_questionario["Q001"] = df_acre["Q001"].map(mapeamento_escolaridade)
df_questionario["Q002"] = df_acre["Q002"].map(mapeamento_escolaridade)



,TP_FAIXA_ETARIA,TP_SEXO,TP_ESTADO_CIVIL,TP_COR_RACA,TP_NACIONALIDADE,TP_ST_CONCLUSAO,TP_ANO_CONCLUIU,TP_ESCOLA,TP_ENSINO,IN_TREINEIRO,CO_MUNICIPIO_ESC,NO_MUNICIPIO_ESC,CO_UF_ESC,SG_UF_ESC,TP_DEPENDENCIA_ADM_ESC,TP_LOCALIZACAO_ESC,TP_SIT_FUNC_ESC,CO_MUNICIPIO_PROVA,CO_UF_PROVA,SG_UF_PROVA,TP_PRESENCA_CN,TP_PRESENCA_CH,TP_PRESENCA_LC,TP_PRESENCA_MT,CO_PROVA_CN,CO_PROVA_CH,CO_PROVA_LC,CO_PROVA_MT,NU_NOTA_CN,NU_NOTA_CH,NU_NOTA_LC,NU_NOTA_MT,TP_LINGUA,TP_STATUS_REDACAO,NU_NOTA_COMP1,NU_NOTA_COMP2,NU_NOTA_COMP3,NU_NOTA_COMP4,NU_NOTA_COMP5,NU_NOTA_REDACAO,Q001,Q002,Q003,Q004,Q005,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017,Q018,Q019,Q020,Q021,Q022,Q023,Q024,Q025
0,6,F,0,3,1,1,0,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1223.0,1194.0,1203.0,1213.0,369.9,387.1,438.3,360.2,1,1.0,100.0,120.0,100.0,80.0,0.0,400.0,4,2,D,B,4,C,A,B,D,A,C,B,A,A,B,B,A,A,B,A,A,E,A,B,B
1,3,M,0,3,1,1,1,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,0,1,1,0,NaN,1192.0,1202.0,NaN,NaN,555.5,549.1,NaN,1,1.0,160.0,120.0,160.0,200.0,100.0,740.0,4,5,E,A,2,C,A,B,B,A,A,B,A,A,B,A,A,A,B,A,A,A,A,A,B
2,7,M,1,2,1,1,5,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200013,12,AC,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,2,A,A,3,B,A,B,C,B,A,B,B,B,A,A,A,A,B,A,A,C,A,B,B
3,7,F,1,3,1,1,5,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,4,A,B,3,E,A,B,D,B,A,B,A,A,A,A,A,A,B,A,A,D,A,A,B
4,6,F,1,3,2,1,1,1,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1200401,12,AC,1,1,1,1,1222.0,1191.0,1201.0,1212.0,331.8,476.9,454.5,342.6,0,1.0,80.0,80.0,80.0,80.0,40.0,360.0,6,2,C,B,2,B,A,B,D,A,A,B,A,B,A,A,A,A,B,A,A,C,A,A,B
